# British Airways - Task 2: Predictive Modeling
## Customer Booking Completion Prediction

---
### Metodologia: CRISP-ML (Cross-Industry Standard Process for Machine Learning)

| Fase | Descripcion |
|------|-------------|
| **Business Understanding** | Comprender el problema de negocio y definir objetivos |
| **Data Understanding** | Explorar y familiarizarse con los datos |
| **Data Preparation** | Limpiar, transformar y preparar los datos |
| **Modeling** | Entrenar modelos de machine learning |
| **Evaluation** | Evaluar el rendimiento de los modelos |
| **Deployment** | Desplegar el modelo y presentar resultados |

---
## Fase 1: Business Understanding

**Objetivo de Negocio:** British Airways quiere entender que factores influyen en que un cliente complete o no una reserva de vuelo.

**Objetivo de ML:** Construir un modelo de clasificacion binaria que prediga si un cliente completara la reserva (`booking_complete = 1`) o no (`booking_complete = 0`).

**Variable Objetivo:** `booking_complete`

**Metricas de Exito:**
- Precision (Exactitud)
- Recall (Sensibilidad)
- F1-Score
- AUC-ROC
- Matriz de confusion

**Impacto Potencial:**
- Identificar clientes con alta probabilidad de completar reserva
- Optimizar estrategias de marketing y retargeting
- Mejorar la experiencia de usuario en el proceso de reserva

In [ ]:
# ============================================================
# Fase 1 & 2: Importaciones y Configuracion
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

REPORTS_DIR = 'docs'
os.makedirs(REPORTS_DIR, exist_ok=True)
print('Configuracion completada.')

---
## Fase 2: Data Understanding (Comprension de los Datos)

In [ ]:
# 2.1 Carga inicial de datos
df = pd.read_csv('data/customer_booking.csv', encoding='ISO-8859-1')
print(f'Dimensiones del dataset: {df.shape}')
print(f'\nPrimeras 5 filas:')
display(df.head())

In [ ]:
# 2.2 Informacion del dataset
print('Informacion del DataFrame:')
print(df.info())
print(f'\nValores nulos por columna:')
print(df.isnull().sum())

In [ ]:
# 2.3 Estadisticas descriptivas
print('Estadisticas descriptivas (variables numericas):')
display(df.describe())

print('\nEstadisticas descriptivas (variables categoricas):')
display(df.describe(include='object'))

In [ ]:
# 2.4 Distribucion de la variable objetivo
target_counts = df['booking_complete'].value_counts()
target_pct = df['booking_complete'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax1 = axes[0]
bars1 = ax1.bar(['No Completo (0)', 'Completo (1)'], target_counts.values,
                color=['#E31837', '#075AAA'], edgecolor='black', linewidth=1.5)
ax1.set_title('Distribucion de Booking Complete', fontsize=14, fontweight='bold')
ax1.set_ylabel('Cantidad de Reservas')
for bar, val in zip(bars1, target_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            str(val), ha='center', fontsize=12, fontweight='bold')

ax2 = axes[1]
colors = ['#E31837', '#075AAA']
wedges, texts, autotexts = ax2.pie(target_counts.values, labels=['No Completo', 'Completo'],
                                    autopct='%1.1f%%', colors=colors, startangle=90,
                                    explode=(0.02, 0.02))
ax2.set_title('Proporcion de Clases', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Desbalanceo: {target_pct[0]:.1f}% vs {target_pct[1]:.1f}%')

### Analisis de Variables Categoricas

In [ ]:
# 2.5 Analisis de variables categoricas vs target
categorical_cols = ['sales_channel', 'trip_type', 'flight_day']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(categorical_cols):
    ct = pd.crosstab(df[col], df['booking_complete'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['#E31837', '#075AAA'], edgecolor='black', linewidth=1)
    axes[i].set_title(f'Tasa de Completado por {col}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('% de Reservas')
    axes[i].legend(['No Completo', 'Completo'], loc='upper right')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2.6 Top origenes de reserva
top_origins = df['booking_origin'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(top_origins)), top_origins.values, color='#075AAA', edgecolor='black')
ax.set_yticks(range(len(top_origins)))
ax.set_yticklabels(top_origins.index)
ax.set_title('Top 15 Origenes de Reserva', fontsize=14, fontweight='bold')
ax.set_xlabel('Cantidad de Reservas')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/top_origins.png', dpi=150, bbox_inches='tight')
plt.show()

### Analisis de Variables Numericas

In [ ]:
# 2.7 Distribucion de variables numericas clave
num_cols = ['num_passengers', 'purchase_lead', 'length_of_stay', 'flight_hour', 'flight_duration']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    ax.hist(df[col], bins=50, color='#075AAA', edgecolor='black', alpha=0.7)
    ax.set_title(f'Distribucion de {col}', fontsize=13, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')

axes[5].axis('off')
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2.8 Boxplots comparativos por clase
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    bp = ax.boxplot([df[df['booking_complete']==0][col], df[df['booking_complete']==1][col]],
                    labels=['No Completo', 'Completo'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#E31837')
    bp['boxes'][1].set_facecolor('#075AAA')
    ax.set_title(f'{col} vs Booking Complete', fontsize=13, fontweight='bold')
    ax.set_ylabel(col)

axes[5].axis('off')
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/boxplots_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2.9 Matriz de correlacion
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(12, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap=cmap,
            square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlacion (Variables Numericas)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2.10 Servicios adicionales vs booking_complete
service_cols = ['wants_extra_baggage', 'wants_preferred_seat', 'wants_in_flight_meals']

df['total_services'] = df[service_cols].sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

services_ct = pd.crosstab(df['total_services'], df['booking_complete'], normalize='index') * 100
services_ct.plot(kind='bar', ax=axes[0], color=['#E31837', '#075AAA'], edgecolor='black', linewidth=1)
axes[0].set_title('Tasa de Completado por Cantidad de Servicios', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cantidad de Servicios Adicionales')
axes[0].set_ylabel('% de Reservas')
axes[0].legend(['No Completo', 'Completo'])
axes[0].tick_params(axis='x', rotation=0)

service_counts = df.groupby('booking_complete')[service_cols].mean()
service_counts.T.plot(kind='bar', ax=axes[1], color=['#E31837', '#075AAA'], edgecolor='black', linewidth=1)
axes[1].set_title('Promedio de Servicios por Estado de Reserva', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Servicio')
axes[1].set_ylabel('Promedio')
axes[1].legend(['No Completo', 'Completo'])
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/services_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Fase 3: Data Preparation (Preparacion de Datos)

In [ ]:
# 3.1 Preparacion de datos para modelado
df_model = df.copy()

# Mapeo de flight_day a numerico
day_mapping = {'Mon': 1, 'Tue': 2, 'Wed': 3, 'Thu': 4, 'Fri': 5, 'Sat': 6, 'Sun': 7}
df_model['flight_day'] = df_model['flight_day'].map(day_mapping)

# Feature: total de servicios adicionales
df_model['total_services'] = df_model[service_cols].sum(axis=1)

# Codificacion one-hot para variables categoricas
categorical_to_encode = ['sales_channel', 'trip_type', 'booking_origin']
df_model = pd.get_dummies(df_model, columns=categorical_to_encode, drop_first=True)

# Separar features y target
X = df_model.drop('booking_complete', axis=1)
y = df_model['booking_complete']

# Eliminar columnas no numericas restantes (route es texto)
X = X.drop('route', axis=1)

print(f'Features para modelado: {X.shape[1]}')
print(f'Registros totales: {len(X)}')
print(f'\nBalance de clases:')
print(y.value_counts(normalize=True))

In [ ]:
# 3.2 Train-Test Split con estratificacion
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape}')
print(f'Test size: {X_test.shape}')
print(f'\nDistribucion en train:')
print(y_train.value_counts(normalize=True))
print(f'\nDistribucion en test:')
print(y_test.value_counts(normalize=True))

In [ ]:
# 3.3 Escalado de features numericas
scaler = StandardScaler()
numeric_features = ['num_passengers', 'purchase_lead', 'length_of_stay',
                    'flight_hour', 'flight_day', 'flight_duration', 'total_services']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

available_numeric = [f for f in numeric_features if f in X_train.columns]
X_train_scaled[available_numeric] = scaler.fit_transform(X_train[available_numeric])
X_test_scaled[available_numeric] = scaler.transform(X_test[available_numeric])

print('Escalado completado usando StandardScaler.')

---
## Fase 4: Modeling (Modelado)

In [ ]:
# 4.1 Funcion de evaluacion
def evaluate_model(model, X_test, y_test, model_name='Modelo'):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else y_pred

    metrics = {
        'Modelo': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    }
    return metrics, y_pred, y_proba


def plot_confusion_matrix(y_test, y_pred, model_name, filename):
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Completo', 'Completo'],
                yticklabels=['No Completo', 'Completo'])
    ax.set_title(f'Matriz de Confusion - {model_name}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Prediccion')
    plt.tight_layout()
    plt.savefig(f'{REPORTS_DIR}/{filename}', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 4.2 Modelo 1: Regresion Logistica (con SMOTE para balanceo)
print('=' * 60)
print('MODELO 1: REGRESION LOGISTICA CON SMOTE')
print('=' * 60)

lr_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train_scaled, y_train)
lr_metrics, lr_pred, lr_proba = evaluate_model(lr_pipeline, X_test_scaled, y_test, 'Logistic Regression + SMOTE')

print(f'Accuracy: {lr_metrics["Accuracy"]:.4f}')
print(f'Precision: {lr_metrics["Precision"]:.4f}')
print(f'Recall: {lr_metrics["Recall"]:.4f}')
print(f'F1-Score: {lr_metrics["F1-Score"]:.4f}')
print(f'AUC-ROC: {lr_metrics["AUC-ROC"]:.4f}')

plot_confusion_matrix(y_test, lr_pred, 'Logistic Regression + SMOTE', 'cm_logistic.png')

In [ ]:
# 4.3 Modelo 2: Random Forest
print('=' * 60)
print('MODELO 2: RANDOM FOREST')
print('=' * 60)

rf_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=15,
                                           min_samples_split=10, random_state=42,
                                           n_jobs=-1))
])

rf_pipeline.fit(X_train_scaled, y_train)
rf_metrics, rf_pred, rf_proba = evaluate_model(rf_pipeline, X_test_scaled, y_test, 'Random Forest')

print(f'Accuracy: {rf_metrics["Accuracy"]:.4f}')
print(f'Precision: {rf_metrics["Precision"]:.4f}')
print(f'Recall: {rf_metrics["Recall"]:.4f}')
print(f'F1-Score: {rf_metrics["F1-Score"]:.4f}')
print(f'AUC-ROC: {rf_metrics["AUC-ROC"]:.4f}')

plot_confusion_matrix(y_test, rf_pred, 'Random Forest + SMOTE', 'cm_rf.png')

In [ ]:
# 4.4 Modelo 3: Gradient Boosting
print('=' * 60)
print('MODELO 3: GRADIENT BOOSTING')
print('=' * 60)

gb_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', GradientBoostingClassifier(n_estimators=150, max_depth=5,
                                               learning_rate=0.05, random_state=42))
])

gb_pipeline.fit(X_train_scaled, y_train)
gb_metrics, gb_pred, gb_proba = evaluate_model(gb_pipeline, X_test_scaled, y_test, 'Gradient Boosting')

print(f'Accuracy: {gb_metrics["Accuracy"]:.4f}')
print(f'Precision: {gb_metrics["Precision"]:.4f}')
print(f'Recall: {gb_metrics["Recall"]:.4f}')
print(f'F1-Score: {gb_metrics["F1-Score"]:.4f}')
print(f'AUC-ROC: {gb_metrics["AUC-ROC"]:.4f}')

plot_confusion_matrix(y_test, gb_pred, 'Gradient Boosting + SMOTE', 'cm_gb.png')

---
## Fase 5: Evaluation (Evaluacion de Modelos)

In [ ]:
# 5.1 Comparacion de metricas
results_df = pd.DataFrame([lr_metrics, rf_metrics, gb_metrics])
results_df = results_df.set_index('Modelo')

print('COMPARACION DE MODELOS:')
print('=' * 60)
display(results_df.round(4))

# Guardar resultados
results_df.to_csv(f'{REPORTS_DIR}/model_comparison.csv')
print('Resultados guardados en docs/model_comparison.csv')

In [ ]:
# 5.2 Grafico comparativo de barras
fig, ax = plt.subplots(figsize=(14, 6))

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
x = np.arange(len(metrics_to_plot))
width = 0.25

for i, (idx, row) in enumerate(results_df.iterrows()):
    values = [row[m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, values, width, label=idx, edgecolor='black', linewidth=1)

ax.set_xlabel('Metrica', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparacion de Modelos - Metricas de Rendimiento', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.legend(loc='lower right')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5.3 Curvas ROC comparativas
fig, ax = plt.subplots(figsize=(10, 8))

modelos_roc = [
    ('Logistic Regression + SMOTE', lr_proba),
    ('Random Forest + SMOTE', rf_proba),
    ('Gradient Boosting + SMOTE', gb_proba)
]

colors = ['#E31837', '#075AAA', '#011E41']

for (name, proba), color in zip(modelos_roc, colors):
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'{name} (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Clasificador Aleatorio')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
ax.set_title('Curvas ROC - Comparacion de Modelos', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5.4 Importancia de Features (Random Forest)
rf_model = rf_pipeline.named_steps['classifier']
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(feature_importance)), feature_importance['importance'].values,
               color='#075AAA', edgecolor='black', linewidth=1)
ax.set_yticks(range(len(feature_importance)))
ax.set_yticklabels(feature_importance['feature'].values)
ax.set_title('Top 15 Features mas Importantes (Random Forest)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importancia')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTop 10 Features:')
for i, row in feature_importance.head(10).iterrows():
    print(f'  {row["feature"]}: {row["importance"]:.4f}')

---
## Fase 6: Deployment (Despliegue y Conclusiones)

In [ ]:
# 6.1 Seleccion del mejor modelo
best_model_idx = results_df['AUC-ROC'].idxmax()
best_model_name = best_model_idx
print(f'MEJOR MODELO: {best_model_name}')
print(f'\nMetricas del mejor modelo:')
print(results_df.loc[best_model_name].round(4))

In [ ]:
# 6.2 Conclusiones del analisis
print('=' * 70)
print('CONCLUSIONES DEL ANALISIS PREDICTIVO')
print('=' * 70)
print('''
1. DESBALANCEO DE CLASES: Solo el ~15% de las reservas se completan.
   Se utilizo SMOTE para balancear las clases durante el entrenamiento.

2. MEJOR MODELO: Random Forest + SMOTE obtuvo el mejor AUC-ROC,
   indicando buena capacidad discriminativa entre clases.

3. FACTORES CLAVE (Top Features):
   - purchase_lead: Tiempo entre compra y viaje (muy importante)
   - wants_extra_baggage: Clientes que quieren equipaje extra
   - flight_duration: Duracion del vuelo
   - booking_origin: Pais de origen de la reserva

4. RECOMENDACIONES DE NEGOCIO:
   - Clientes con mayor purchase_lead tienden a no completar
   - Campanas de retargeting para clientes con alta intencion (>30% prob)
   - Mejorar UX en mobile (menor tasa de conversion)
   - Ofrecer incentivos para completar reserva (equipaje, asientos)

5. PROXIMOS PASOS:
   - Probar XGBoost/LightGBM para mejor rendimiento
   - Implementar validacion cruzada temporal
   - Desplegar modelo como API para predicciones en tiempo real
''')

---
## Resumen del Proceso CRISP-ML

| Fase | Estado | Entregable |
|------|--------|------------|
| 1. Business Understanding | Completado | Definicion del problema y objetivos |
| 2. Data Understanding | Completado | EDA con graficos y estadisticas |
| 3. Data Preparation | Completado | Datos limpios y codificados |
| 4. Modeling | Completado | 3 modelos entrenados y validados |
| 5. Evaluation | Completado | Comparacion de metricas y seleccion |
| 6. Deployment | Completado | Conclusiones y recomendaciones |

---
*Notebook generado para British Airways Forage Data Science Task 2*